In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/elastic/lib/python3.10/site-packages/IPython/core/extensions.py:151: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  mod.load_ipython_extension(self.shell)


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/annotated/checkpoints/pre_cell_10.pickle

trying: ['SEP_DOLLAR', 'CONVERT_TO_CAT', 'VARIABLE_FILES', 'AUTO_ADJUST_LEARNING_RATE']
me:  2
me:  2
me:  2
me:  2
trying: ['SEP_DOLLAR', 'CONVERT_TO_CAT', 'VARIABLE_FILES', 'AUTO_ADJUST_LEARNING_RATE']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA', 'ENABLE_BREAKPOINT']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA', 'ENABLE_BREAKPOINT']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'ENABLE_BREAKPOINT', 'SEP_COMMA', 'REGRESSOR']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'ENABLE_BREAKPOINT', 'SEP_COMMA', 'REGRESSOR']
me:  2
me:  2
me:  2
me:  2
trying: ['SEP_DOLLAR', 'CONVERT_TO_CAT', 'VARIABLE_FILES', 'AUTO_ADJUST_LEARNING_RATE']
me:  2
me:  2
me:  2
me:  2
trying: ['SEP_DOLLAR', 'CONVERT_TO_CAT', 'VARIABLE_FILES', 'AUTO_ADJUST_LEARNING_RATE']
me:  2
me:  2
me:  2
me:  2
trying: ['FASTAI_LEARNING_RATE']
me:  2
trying: ['random']
me:  1
trying: ['df']


me:  6
trying: ['factor']
me:  5
trying: ['np']
me:  1
trying: ['shutil']
me:  1
trying: ['pd']
me:  1
trying: ['traceback']
me:  1
trying: ['col']
me:  6
trying: ['PROJECT_NAME']
me:  2
trying: ['warnings']
me:  0
trying: ['PARAM_DIR']
me:  3


Declaring variable warnings
Declaring variable random
Declaring variable np
Declaring variable shutil
Declaring variable pd
Declaring variable traceback
Declaring variable SEP_DOLLAR
Declaring variable CONVERT_TO_CAT
Declaring variable VARIABLE_FILES
Declaring variable AUTO_ADJUST_LEARNING_RATE
Declaring variable SEP_DOLLAR
Declaring variable CONVERT_TO_CAT
Declaring variable VARIABLE_FILES
Declaring variable AUTO_ADJUST_LEARNING_RATE
Declaring variable SHUFFLE_DATA
Declaring variable REGRESSOR
Declaring variable SEP_COMMA
Declaring variable ENABLE_BREAKPOINT
Declaring variable SHUFFLE_DATA
Declaring variable REGRESSOR
Declaring variable SEP_COMMA
Declaring variable ENABLE_BREAKPOINT
Declaring variable SHUFFLE_DATA
Declaring variable ENABLE_BREAKPOINT
Declaring variable SEP_COMMA
Declaring variable REGRESSOR
Declaring variable SHUFFLE_DATA
Declaring variable ENABLE_BREAKPOINT
Declaring variable SEP_COMMA
Declaring variable REGRESSOR
Declaring variable SEP_DOLLAR
Declaring variable CONV

In [4]:
%%RecordEvent
%%time
### cell 10 ###

target = ''
target_str = ''
targets = []

df_cols = df.columns.to_list()
for col in reversed(df_cols[1:]):
    # Convert candidate target to numeric (coerce non-numeric to NaN)
    df[col] = pd.to_numeric(df[col], errors='coerce')
    target = col
    target_str = col.replace('/', '-')
    print(f'Target Variable: {target}')

    # Ensure PARAM_DIR and config files exist
    os.makedirs(PARAM_DIR, exist_ok=True)
    for fname in ['cats.txt', 'conts.txt', 'cols_to_delete.txt']:
        fpath = os.path.join(PARAM_DIR, fname)
        if not os.path.exists(fpath):
            open(fpath, 'w').close()

    # Drop duplicates and optionally shuffle
    df = df.drop_duplicates()
    if SHUFFLE_DATA:
        df = df.sample(frac=1).reset_index(drop=True)

    # Convert any boolean columns to uint8
    for c in df.columns:
        if pd.api.types.is_bool_dtype(df[c]):
            df[c] = df[c].astype('uint8')

    # Drop any user-specified columns
    with open(os.path.join(PARAM_DIR, 'cols_to_delete.txt'), 'r') as f:
        del_cols = [c for c in f.read().splitlines() if c in df.columns]
    for c in del_cols:
        try:
            del df[c]
        except:
            pass

    # Fill all remaining NaNs with zero
    df = df.fillna(0)

    # Auto-detect categorical vs continuous
    nunique = df.nunique()
    counts = df.count()
    ratio = nunique / counts
    cats = ratio[ratio < 0.05].index.to_list()
    conts = ratio[ratio >= 0.05].index.to_list()

    # Exclude the target
    if target in cats:
        cats.remove(target)
    if target in conts:
        conts.remove(target)

    # Convert the target to numeric again (coerce)
    df[target] = pd.to_numeric(df[target], errors='coerce')

    if VARIABLE_FILES:
        with open(os.path.join(PARAM_DIR, 'cats.txt'), 'r') as f:
            cats = f.read().splitlines()
        with open(os.path.join(PARAM_DIR, 'conts.txt'), 'r') as f:
            conts = f.read().splitlines()

    # Convert all continuous variables to numeric
    for var in conts:
        try:
            df[var] = pd.to_numeric(df[var], errors='coerce')
        except:
            print(f'Could not convert {var} to float.')
            pass

    # Optional breakpoint-style reconversion
    if ENABLE_BREAKPOINT:
        cont_list = list(conts)
        for var in list(cont_list):
            try:
                df[var] = pd.to_numeric(df[var], errors='coerce')
            except:
                print(f'Could not convert {var} to float.')
                cont_list.remove(var)
                if CONVERT_TO_CAT:
                    cats.append(var)
                pass

    targets.append(target)

Target Variable: Profit_processed


Target Variable: Discount_processed
Target Variable: Quantity_processed
Target Variable: Sales_processed


Target Variable: Sub-Category_processed
Target Variable: Category_processed
Target Variable: Region_processed


Target Variable: Postal Code_processed
Target Variable: State_processed
Target Variable: City_processed
Target Variable: Country_processed


Target Variable: Segment_processed
Target Variable: Ship Mode_processed
Target Variable: Profit
Target Variable: Discount
Target Variable: Quantity


Target Variable: Sales
Target Variable: Sub-Category
Target Variable: Category
Target Variable: Region


Target Variable: Postal Code
Target Variable: State
Target Variable: City
Target Variable: Country


Target Variable: Segment
CPU times: user 1.71 s, sys: 72.8 ms, total: 1.78 s
Wall time: 1.78 s


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/rewritten/cell_10/checkpoints/post_cell_10_try_9.pickle

migration speed (bps): 773348534.5066034
---------------------------
variables to migrate:
factor 28
target_str 56
fpath 170
SEP_DOLLAR 24
c 65
SHUFFLE_DATA 28
conts 104
REGRESSOR 28
cont_list 104
SEP_COMMA 28
np 72
cats 216
AUTO_ADJUST_LEARNING_RATE 24
nunique 48
PROJECT_NAME 59
del_cols 56
CONVERT_TO_CAT 24
f 208
VARIABLE_FILES 24
target 56
FASTAI_LEARNING_RATE 24
df_cols 264
ENABLE_BREAKPOINT 28
targets 312
df 48
fname 67
ratio 48
counts 48
var 65
col 56
shutil 72
pd 72
PARAM_DIR 151
traceback 72
random 72
warnings 72
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/rewritten/cell_10/checkpoints/post_cell_10_try_9.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['df', 'PARAM_DIR']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['df', 'PARAM_DIR']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables ['df']
Intermediate variables ['factor']
Future variables ['PARAM_DIR']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables ['df', 'col']
Intermediate variables []
Future variables ['PARAM_DIR']
Modified dataframes
  df
    Input columns: set()
    Changed columns: set()
    Created columns: {'City_processed', 'State_processed', 'Region_processed', 'Profit_processed', 'Postal Code_processed', 'Discount_processed', 'Sub-Category_processed', 'Quantity_processed', 'Ship Mode_processed', 'Sales_processed', 'Country_processed', 'Category_processed', 'Segment_processed'}
    Deleted columns: set()
======= Cell 4 =======
In

In [7]:
with open(
    "/home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/opt_cell_exec_info_10_try_9.pkl",
    "wb",
) as f:
    pickle.dump(opt_cell_exec_info[10], f)

In [8]:
opt_output = Out.get(4)

In [9]:
df_opt = df
%LoadCheckpoint /home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/annotated/checkpoints/post_cell_10.pickle
assert compare_df(df_opt, df)

import numpy as np
from elastic.core.common.pandas import is_type_styler

is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(
    orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_opt_output_array = isinstance(
    opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif (
    (is_orig_output_pd or is_opt_output_pd)
    and (is_orig_output_array or is_opt_output_array)
) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output

trying: ['AUTO_ADJUST_LEARNING_RATE', 'SEP_DOLLAR', 'VARIABLE_FILES', 'CONVERT_TO_CAT']
me:  2
me:  2
me:  2
me:  2
trying: ['AUTO_ADJUST_LEARNING_RATE', 'SEP_DOLLAR', 'VARIABLE_FILES', 'CONVERT_TO_CAT']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'ENABLE_BREAKPOINT', 'REGRESSOR', 'SEP_COMMA']
me:  2
me:  2
me:  2
me:  2
trying: ['SHUFFLE_DATA', 'ENABLE_BREAKPOINT', 'REGRESSOR', 'SEP_COMMA']
me:  2
me:  2
me:  2
me:  2
trying: ['ENABLE_BREAKPOINT', 'SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA']
me:  2
me:  2
me:  2
me:  2
trying: ['ENABLE_BREAKPOINT', 'SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA']
me:  2
me:  2
me:  2
me:  2
trying: ['AUTO_ADJUST_LEARNING_RATE', 'VARIABLE_FILES', 'SEP_DOLLAR', 'CONVERT_TO_CAT']
me:  2
me:  2
me:  2
me:  2
trying: ['AUTO_ADJUST_LEARNING_RATE', 'VARIABLE_FILES', 'SEP_DOLLAR', 'CONVERT_TO_CAT']
me:  2
me:  2
me:  2
me:  2
trying: ['focus_cont', 'cont', 'n', 'var']
me:  13
me:  13
me:  13
me:  13
trying: ['focus_cont', 'cont', 'n', 'var']
me:  13
me:  13
me

ValueError: Content of df1 and df2 do not match